In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
from Datacharacterization_functions import analyze_csv_ranges 

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

## $\text{\textcolor{green}{Day-long Horizon Solver w/ Sized Up Generation, Self-Sufficiency, \& Curtailment}}$


In [ ]:
def solve_battery_day(day_df, load_profile, day_ahead_price, minimum_self_sufficiency, charge_eff, discharge_eff, dT, C_rate):
    day_df = day_df.sort_index()

    PV_gen = day_df["PV_total"].to_numpy()
    Wind_gen = day_df["Wind_total"].to_numpy()
    Pgen_total_np = PV_gen + Wind_gen

    n = len(day_df)

    Pload_np = load_profile
    Pprice_np = day_ahead_price

    # Removing NaN's from the input data
    if len(Pload_np) != n:
        return {"status": "bad_input: load length mismatch"}
    if len(Pprice_np) != n:
        return {"status": "bad_input: price length mismatch"}
    if not np.isfinite(Pload_np).all():
        return {"status": "bad_input: non-finite load"}
    if not np.isfinite(Pprice_np).all():
        return {"status": "bad_input: non-finite price"}
    if not np.isfinite(PV_gen).all():
        return {"status": "bad_input: non-finite PV"}
    if not np.isfinite(Wind_gen).all():
        return {"status": "bad_input: non-finite wind"}

    # Constants
    Pload = cp.Constant(Pload_np)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)

    # Variables
    P_capacity = cp.Variable(nonneg=True)
    E_capacity = cp.Variable(nonneg=True)

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)

    Pgrid_buy = cp.Variable(n, nonneg=True)
    SOC = cp.Variable(n)
    Pcurtail = cp.Variable(n, nonneg=True)

    constraints = [
        P_capacity <= C_rate * E_capacity,

        Pcharge <= P_capacity,
        Pdischarge <= P_capacity,

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        Pcurtail <= Pgen_PV + Pgen_W,

        # Power balance
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge + Pcurtail,

        SOC[0] == 0.1 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,

        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[-1] >= 0.1 * E_capacity,
        
        #Cap on import from the grid based on difference of self-sufficiency
        cp.sum(Pgrid_buy) * dT <= (1 - minimum_self_sufficiency) * np.sum(Pload_np) * dT
    ]

    # Penalization for importing
    E_grid_cost = cp.sum(cp.multiply(Pgrid_buy, Pprice_np)) * dT

    # Penalization for curtailment
    lambda_curt = 1000
    E_curtailed_penalty = cp.sum(Pcurtail * lambda_curt) * dT

    #Penalization for simultaneous charge/discharge <-- RECONSIDERA CÓMO ESTÁS EVITANDO LA CARGA/DESCARGA SIMULTÁNEA; REVISA QUE UNA SEA 0; REFIÉRETE A LA FOTO DE LA PIZARRA 
    lambda_throughput = 3000
    E_charge_discharge = (lambda_throughput * cp.sum(Pcharge + Pdischarge) * dT)

    problem = cp.Problem(cp.Minimize(E_capacity + E_grid_cost + E_curtailed_penalty +  E_charge_discharge), constraints)

    try:
        problem.solve(solver=cp.HIGHS, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            return {
                "status": problem.status,
                "E_capacity_kWh": np.nan,
                "P_capacity_kW": np.nan,
                "total_generation_kWh": np.nan,
                "total_load_kWh": np.nan,
                "Pload_profile": None,
                "Pgen_profile": None,
                "PV_profile": None,
                "Wind_profile": None,
                "Pcharge_profile": None,
                "Pdischarge_profile": None,
                "Pgrid_profile": None,
                "Pcurtail_profile": None,
                "SOC_profile": None,
                "renewable_generation_pct": np.nan,
                "grid_dependence_pct": np.nan,
                "grid_import_kWh": np.nan,
                "peak_grid_import_kW": np.nan,
                "spilled_energy_kWh": np.nan,
                "grid_and_curtail_simultaneous": np.nan,
            }

        total_generation_kWh = np.sum(Pgen_total_np) * dT
        total_load_kWh = np.sum(Pload_np) * dT

        grid_import_kWh = np.sum(Pgrid_buy.value) * dT
        peak_grid_import_kW = np.max(Pgrid_buy.value)

        spilled_energy_kWh = np.sum(Pcurtail.value) * dT

        # Your requested metric: generation / load * 100 => doesn't include battery discharge
        renewable_generation_pct = (total_generation_kWh / total_load_kWh) * 100

        # More operational metric: percentage of imports used to satisfy the load
        grid_dependence_pct = (grid_import_kWh / total_load_kWh) * 100

        both_active = np.any(
            (Pgrid_buy.value > 1e-6) & (Pcurtail.value > 1e-6)
        )

        return {
            "status": problem.status,
            "E_capacity_kWh": E_capacity.value,
            "P_capacity_kW": P_capacity.value,

            "total_generation_kWh": total_generation_kWh,
            "total_load_kWh": total_load_kWh,
            
            #Profiles
            "Pload_profile": Pload_np,
            "Pgen_profile": Pgen_total_np,
            "PV_profile": PV_gen,
            "Wind_profile": Wind_gen,
            "Pcharge_profile": Pcharge.value,
            "Pdischarge_profile": Pdischarge.value,
            "Pgrid_profile": Pgrid_buy.value,
            "Pcurtail_profile": Pcurtail.value,
            "SOC_profile": SOC.value,
            
            "renewable_generation_pct": renewable_generation_pct,
            "grid_dependence_pct": grid_dependence_pct,

            "grid_import_kWh": grid_import_kWh,
            "peak_grid_import_kW": peak_grid_import_kW,

            "spilled_energy_kWh": spilled_energy_kWh,
            "grid_and_curtail_simultaneous": both_active,
        }

    except Exception as e:
        return {
            "status": f"solver_error: {type(e).__name__}: {str(e)}",
            "E_capacity_kWh": np.nan,
            "P_capacity_kW": np.nan,
            "total_generation_kWh": np.nan,
            "total_load_kWh": np.nan,
            "Pload_profile": None,
            "Pgen_profile": None,
            "PV_profile": None,
            "Wind_profile": None,
            "Pcharge_profile": None,
            "Pdischarge_profile": None,
            "Pgrid_profile": None,
            "Pcurtail_profile": None,
            "SOC_profile": None,
            "Pgrid_profile": None,
            "Pcurtail_profile": None,
            "SOC_profile": None,
            "renewable_generation_pct": np.nan,
            "grid_dependence_pct": np.nan,
            "grid_import_kWh": np.nan,
            "peak_grid_import_kW": np.nan,
            "spilled_energy_kWh": np.nan,
            "grid_and_curtail_simultaneous": np.nan,
        }

$\text{\textcolor{green}{Load}}$

In [ ]:
df_load = pd.read_csv(ROOT / "data/archive/archive/building_consumption.csv",parse_dates=["timestamp"])

filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14 07:30:00") &
    (df_load["timestamp"] <= "2022-04-10 23:30:00")
]

total_load = (
    filtered
    .groupby("timestamp")["consumption"]
    .sum()
    .reset_index(name="total_consumption")
)

total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean()

load_scale = 0.2
total_load_30min["total_consumption"] *= load_scale

total_load_30min["date"] = total_load_30min.index.date
total_load_30min["month_day"] = total_load_30min.index.strftime("%m-%d")

load_counts = total_load_30min.groupby("date")["total_consumption"].count()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[total_load_30min["date"].isin(load_complete_days)].copy()

load_day_profiles = []

for date, day_df in load_days_df.groupby("date"):
    day_df = day_df.sort_index()

    load_day_profiles.append({
        "load_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "Load_profile": day_df["total_consumption"].to_numpy()
    })

load_profiles_df = pd.DataFrame(load_day_profiles)


$\text{\textcolor{green}{Generation}}$

In [ ]:
DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"

df_gen = pd.read_csv(DATA_DIR, delimiter=",", decimal=".", parse_dates=["ts"], index_col="ts")

df_agg_gen = df_gen.copy()

df_agg_gen["PV_total"] = df_agg_gen[["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]].sum(axis=1, min_count=1)
df_agg_gen["PV_total"] *= 15

df_agg_gen["Wind_total"] = df_agg_gen[["Aircon_WT Power_m", "Gaia_WT Power_m"]].sum(axis=1, min_count=1)
df_agg_gen["Wind_total"] *= 2
df_agg_gen.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)

df_agg_gen["date"] = df_agg_gen.index.date
df_agg_gen["month_day"] = df_agg_gen.index.strftime("%m-%d")

counts_per_day = df_agg_gen.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index

df_agg_gen = df_agg_gen[df_agg_gen["date"].isin(complete_days)].copy()

gen_day_profiles = []

for date, day_df in df_agg_gen.groupby("date"):
    day_df = day_df.sort_index()

    pv_profile = day_df["PV_total"].to_numpy()
    wind_profile = day_df["Wind_total"].to_numpy()
    renewable_profile = pv_profile + wind_profile

    gen_day_profiles.append({
        "gen_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "PV_profile": pv_profile,
        "Wind_profile": wind_profile,
        "Renewable_profile": renewable_profile
    })

gen_profiles_df = pd.DataFrame(gen_day_profiles)

$\text{\textcolor{green}{Day-Ahead Prices}}$

In [ ]:
df_price = pd.read_csv(ROOT / "data/DayAheadData_fulltimeperiod.csv")

df_price = df_price[(df_price["ts"] >= "2025-05-14 07:30:00") &
    (df_price["ts"] <= "2026-04-10 23:30:00")]

df_price["ts"] = pd.to_datetime(df_price["ts"])
df_price = df_price.set_index("ts")

df_price["date"] = df_price.index.date
df_price["month_day"] = df_price.index.strftime("%m-%d")

counts_per_day_price = df_price.groupby("date").size()
complete_days_price = counts_per_day_price[counts_per_day_price == 48].index

df_price = df_price[df_price["date"].isin(complete_days_price)].copy()

price_day_profiles = []

for date, day_df in df_price.groupby("date"):
    day_df = day_df.sort_index()
    
    price_day_profiles.append({
        "gen_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "Price_profile": day_df["DayAheadPriceDKK"].to_numpy()
    })

price_profiles_df = pd.DataFrame(price_day_profiles)


In [ ]:
paired_day_profiles = load_profiles_df.merge(gen_profiles_df, on="month_day", how="inner").merge(price_profiles_df, on="month_day", how="inner")

paired_day_profiles = (paired_day_profiles.sort_values("month_day").reset_index(drop=True))

# Load power statistics
all_load_values = np.concatenate(paired_day_profiles["Load_profile"].to_numpy())

mean_power_kW = np.mean(all_load_values)
min_power_kW = np.min(all_load_values)
max_power_kW = np.max(all_load_values)

print(f"Mean load power: {mean_power_kW:.2f} kW")
print(f"Minimum load power: {min_power_kW:.2f} kW")
print(f"Maximum load power: {max_power_kW:.2f} kW")


In [ ]:
# Visualize load
time_labels = pd.date_range("00:00", periods=48, freq="30min").strftime("%H:%M")
    # Power of load
plt.figure(figsize=(10, 6))

for profile in paired_day_profiles["Load_profile"]:
    plt.plot(time_labels, profile, alpha=0.25)

plt.xticks(time_labels[::4], rotation=45)
plt.xlabel("Time of day")
plt.ylabel("Load [kW]")
plt.title("Daily load profiles")
plt.grid(True)
plt.show()

    # Energy of load across all paired days 

total_energy_kWh = sum(
sum(profile) * 0.5
    for profile in paired_day_profiles["Load_profile"])
print(f"Total energy demand: {total_energy_kWh:.0f} kWh")

n_days = len(paired_day_profiles)
avg_daily_energy = total_energy_kWh / n_days
print(f"Average daily energy: {avg_daily_energy:.1f} kWh/day")

avg_power_kW = avg_daily_energy / 24
print(f"Equivalent constant load: {avg_power_kW:.1f} kW")

# Visualize generation 

plt.figure(figsize=(10, 6))

for profile in paired_day_profiles["Renewable_profile"]:
    plt.plot(time_labels, profile, alpha=0.25)

plt.xticks(time_labels[::4], rotation=45)
plt.xlabel("Time of day")
plt.ylabel("Generation [kW]")
plt.title("Daily generation profiles")
plt.grid(True)
plt.show()


$\text{\textcolor{green}{Net Load and Self-Sufficiency Diagnostic}}$

In [ ]:
diagnostics = []
dT = 0.5  # still needed for cumulative behavior

for _, row in paired_day_profiles.iterrows():
    load_profile = row["Load_profile"]
    gen_profile = row["Renewable_profile"]

    net_load = load_profile - gen_profile
    cumulative = np.cumsum(net_load * dT)

    # --- Power-based metrics ---
    avg_load_kW = load_profile.mean()
    avg_gen_kW = gen_profile.mean()

    peak_load_kW = load_profile.max()
    peak_gen_kW = gen_profile.max()

    # Direct overlap in power (instantaneous matching)
    direct_used_profile = np.minimum(load_profile, gen_profile)
    avg_direct_used_kW = direct_used_profile.mean()

    # Mismatch metrics
    mean_abs_mismatch_kW = np.mean(np.abs(net_load))

    diagnostics.append({
        "load_date": row["load_date"],
        "gen_date": row["gen_date_x"],
        "month_day": row["month_day"],

        # --- Power comparison ---
        "avg_load_kW": avg_load_kW,
        "avg_gen_kW": avg_gen_kW,
        "avg_direct_used_kW": avg_direct_used_kW,

        "peak_load_kW": peak_load_kW,
        "peak_gen_kW": peak_gen_kW,

        "max_deficit_kW": np.maximum(net_load, 0).max(),
        "max_surplus_kW": np.maximum(-net_load, 0).max(),

        "mean_abs_mismatch_kW": mean_abs_mismatch_kW,

        # Still useful (links to storage need)
        "ideal_storage_swing_kWh": cumulative.max() - cumulative.min()
    })

diagnostics_df = pd.DataFrame(diagnostics)

diagnostics_df.describe()

### $\text{\textcolor{green}{Optimization With Different Self-Sufficiency Targets \& Sized Up Generation}}$

$\text{\textcolor{green}{C-rate can remain fixed at 0.5 since it doesn't affect the size of the battery}}$

In [ ]:
targets = np.linspace(0.1, 1.0, 10)

results = []
skipped_days = 0

for target in targets:
    for _, row in paired_day_profiles.iterrows():

        load_profile = np.asarray(row["Load_profile"])
        pv_profile = np.asarray(row["PV_profile"])
        wind_profile = np.asarray(row["Wind_profile"])
        price_profile = np.asarray(row["Price_profile"])

        day_df = pd.DataFrame({
            "PV_total": pv_profile,
            "Wind_total": wind_profile
        })

        if not (len(day_df) == len(load_profile) == len(price_profile) == 48):
            skipped_days += 1
            continue

        out = solve_battery_day(
            day_df,
            load_profile=load_profile,
            day_ahead_price=price_profile,
            minimum_self_sufficiency=target,
            charge_eff=0.95,
            discharge_eff=0.95,
            dT=0.5,
            C_rate=0.5,
        )

        out["is_feasible"] = out["status"] in ["optimal", "optimal_inaccurate"]
        out["load_date"] = row["load_date"]
        out["gen_date_x"] = row["gen_date_x"]
        out["month_day"] = row["month_day"]
        out["target_self_sufficiency"] = target
        out["day_id"] = f"{row['load_date']}_{row['gen_date_x']}"
        out["C_rate"] = 0.5

        results.append(out)

optimization_results = pd.DataFrame(results)

optimization_results.columns
print(f"Skipped days: {skipped_days}")

$\text{\textcolor{green}{Summary of Optimization Results}}$

In [ ]:
optimization_results["is_feasible"] = (
    optimization_results["status"].isin(["optimal", "optimal_inaccurate"])
)

optimization_results["storage_duration_hrs"] = (
    optimization_results["E_capacity_kWh"] / optimization_results["P_capacity_kW"]
)

summary = (
    optimization_results
    .groupby("target_self_sufficiency")
    .agg(
        median_E_capacity_kWh=("E_capacity_kWh", "median"),
        p90_E_capacity_kWh=("E_capacity_kWh", lambda x: x.quantile(0.90)),
        max_E_capacity_kWh=("E_capacity_kWh", "max"),

        median_P_capacity_kW=("P_capacity_kW", "median"),
        p90_P_capacity_kW=("P_capacity_kW", lambda x: x.quantile(0.90)),
        max_P_capacity_kW=("P_capacity_kW", "max"),

        median_storage_duration_hrs=("storage_duration_hrs", "median"),
        p90_storage_duration_hrs=("storage_duration_hrs", lambda x: x.quantile(0.90)),
        max_storage_duration_hrs=("storage_duration_hrs", "max"),

        median_grid_import_kWh=("grid_import_kWh", "median"),
        p90_grid_import_kWh=("grid_import_kWh", lambda x: x.quantile(0.90)),
        max_grid_import_kWh=("grid_import_kWh", "max"),

        median_peak_grid_import_kW=("peak_grid_import_kW", "median"),
        p90_peak_grid_import_kW=("peak_grid_import_kW", lambda x: x.quantile(0.90)),
        max_peak_grid_import_kW=("peak_grid_import_kW", "max"),

        median_spilled_energy_kWh=("spilled_energy_kWh", "median"),
        p90_spilled_energy_kWh=("spilled_energy_kWh", lambda x: x.quantile(0.90)),
        max_spilled_energy_kWh=("spilled_energy_kWh", "max"),

        feasible_days=("is_feasible", "sum"),
        total_days=("is_feasible", "count")
    )
    .reset_index()
)

summary["feasibility_rate"] = (
    summary["feasible_days"] / summary["total_days"]
) * 100

summary["average_load_kW"] = avg_power_kW

summary["target_self_sufficiency"] = (
    summary["target_self_sufficiency"] * 100
).round(0).astype(int)

summary.round(2)

Even with generation in the gigagwatt scale, it's very hard to achieve 100% self-sufficiency on every day of the year. The PV generation has been upscaled by a factor of 30––210 panels––and the generation by a factor of 2–––2 turbines. PV was easier to upscale than the turbines since it has a larger output and solar panels can be easily installed in large numbers as self-standinga rrays or those installed in roofs.

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>target_self_sufficiency</th>
      <th>median_E_capacity_kWh</th>
      <th>p90_E_capacity_kWh</th>
      <th>max_E_capacity_kWh</th>
      <th>median_P_capacity_kW</th>
      <th>p90_P_capacity_kW</th>
      <th>max_P_capacity_kW</th>
      <th>median_storage_duration_hrs</th>
      <th>p90_storage_duration_hrs</th>
      <th>max_storage_duration_hrs</th>
      <th>...</th>
      <th>median_peak_grid_import_kW</th>
      <th>p90_peak_grid_import_kW</th>
      <th>max_peak_grid_import_kW</th>
      <th>median_spilled_energy_kWh</th>
      <th>p90_spilled_energy_kWh</th>
      <th>max_spilled_energy_kWh</th>
      <th>feasible_days</th>
      <th>total_days</th>
      <th>feasibility_rate</th>
      <th>average_load_kW</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>10</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>264.97</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>132.49</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>114.89</td>
      <td>137.88</td>
      <td>161.73</td>
      <td>3544.26</td>
      <td>9693.51</td>
      <td>13340.62</td>
      <td>321</td>
      <td>322</td>
      <td>99.69</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>1</th>
      <td>20</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>264.97</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>132.49</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>114.77</td>
      <td>138.05</td>
      <td>161.73</td>
      <td>3639.42</td>
      <td>9700.05</td>
      <td>13340.62</td>
      <td>318</td>
      <td>322</td>
      <td>98.76</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>2</th>
      <td>30</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>264.97</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>132.49</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>114.38</td>
      <td>137.49</td>
      <td>161.73</td>
      <td>4266.49</td>
      <td>9782.79</td>
      <td>13340.62</td>
      <td>297</td>
      <td>322</td>
      <td>92.24</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>3</th>
      <td>40</td>
      <td>0.00</td>
      <td>247.81</td>
      <td>396.27</td>
      <td>0.00</td>
      <td>123.90</td>
      <td>198.14</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>114.17</td>
      <td>135.53</td>
      <td>161.73</td>
      <td>4841.29</td>
      <td>10113.59</td>
      <td>13340.62</td>
      <td>273</td>
      <td>322</td>
      <td>84.78</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>4</th>
      <td>50</td>
      <td>0.00</td>
      <td>446.09</td>
      <td>632.78</td>
      <td>0.00</td>
      <td>223.05</td>
      <td>316.39</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>113.24</td>
      <td>134.00</td>
      <td>161.73</td>
      <td>5282.58</td>
      <td>10301.71</td>
      <td>13340.62</td>
      <td>254</td>
      <td>322</td>
      <td>78.88</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>5</th>
      <td>60</td>
      <td>222.42</td>
      <td>718.20</td>
      <td>953.60</td>
      <td>111.21</td>
      <td>359.10</td>
      <td>476.80</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>111.13</td>
      <td>126.82</td>
      <td>161.73</td>
      <td>5450.48</td>
      <td>10546.11</td>
      <td>13340.62</td>
      <td>241</td>
      <td>322</td>
      <td>74.84</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>6</th>
      <td>70</td>
      <td>421.15</td>
      <td>916.65</td>
      <td>1274.43</td>
      <td>210.58</td>
      <td>458.32</td>
      <td>637.21</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>106.49</td>
      <td>118.32</td>
      <td>161.73</td>
      <td>5525.70</td>
      <td>10504.05</td>
      <td>13149.18</td>
      <td>226</td>
      <td>322</td>
      <td>70.19</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>7</th>
      <td>80</td>
      <td>560.73</td>
      <td>889.66</td>
      <td>1144.23</td>
      <td>280.37</td>
      <td>444.83</td>
      <td>572.12</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>102.06</td>
      <td>111.99</td>
      <td>134.83</td>
      <td>6604.44</td>
      <td>11209.02</td>
      <td>12864.99</td>
      <td>147</td>
      <td>322</td>
      <td>45.65</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>8</th>
      <td>90</td>
      <td>440.00</td>
      <td>552.09</td>
      <td>552.93</td>
      <td>220.00</td>
      <td>276.05</td>
      <td>276.46</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>75.94</td>
      <td>101.57</td>
      <td>107.18</td>
      <td>9652.46</td>
      <td>11472.27</td>
      <td>11506.18</td>
      <td>4</td>
      <td>322</td>
      <td>1.24</td>
      <td>107.91</td>
    </tr>
    <tr>
      <th>9</th>
      <td>100</td>
      <td>605.43</td>
      <td>605.43</td>
      <td>605.43</td>
      <td>302.71</td>
      <td>302.71</td>
      <td>302.71</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>...</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>0.00</td>
      <td>11122.57</td>
      <td>11122.57</td>
      <td>11122.57</td>
      <td>1</td>
      <td>322</td>
      <td>0.31</td>
      <td>107.91</td>
    </tr>
  </tbody>
</table>
<p>10 rows × 23 columns</p>
</div>

Notes on table above:
- renewable_generation_pct does not consider the fraction serving the load that comes from the battery because the latter can be charged from the renewable surplus (self-sufficiency) and from the grid imports (system-dependent). 
- renewable_generation_pct + grid_dependence_pct > 100 $\%$ because grid imports also go to charge the battery, i.e. may be larger than the load to meet constraints like minimum SoC.
- storage_duration_hrs is how long the battery can (dis)charge at maximum power before emptying.

CAPEX comes from the cumulative price of the following elements:
1. battery cells (Table 11, DEA-Technology Catalogue)
2. packaging of the cells into batter packs with BMS (Table 11, DEA_Technology Catalogue)
3. Procurement of battery packs in racks within containers, which includes thermal management systems (TMS) and a fire suppression system (FSS)*
4. Labour for installation and assembly*
5. Balance of System (BOS), which includes wiring and electrical support equipment*
6. Power Conversion System (PCS), which converts bidirectionally between DC current for the battery system and the AC current for the grid*
7. Cost of connecting the BESS t the grid, which includes a transformer*
8. Cost of engineering, procurement, and constructions (EPC), which encompasses site-specific engineering work, procurement of materials and construction equipment, as well as siting, installation, and comissioning of the BESS*
    
    *The costs of items 3-8 come from Table 13  DEA-Technology Catalogue


Fixed $O\&M$ as a percentage of the CAPEX comes from the Danish Energy Agency's linearly interpolated formula in the Technology Catalogue for Energy Storage Technologies

$1.43\% * \frac{P}{E} + 3.14\%$



In [ ]:
#CAPEX of battery with imports as need arises
battery_cell_cost = 0.084 * 1000 #EUR/kWh -- 2020 EUR
pack_cost = 0.025 * 1000 #EUR/kWh -- 2020 EUR
container_TMS_FSS_cost = 0.081 * 1000 #EUR/kWh --2025 EUR
labor_installation_cost = 0.010 * 1000 #EUR/kWh -- 2025 EUR
PCS = 0.069 * 1000 #EUR/kW -- 2025 EUR
BOS = 0.037 * 1000 #EUR/kWh -- 2025 EUR
grid_connection = 0.017 * 1000 #EUR/kW -- 2025 EUR
# EPC = # 10% of the sum the above -- 2025 EUR

CAPEX_energy_capacity = (battery_cell_cost + pack_cost + container_TMS_FSS_cost + labor_installation_cost + BOS) * summary["E_capacity_kWh"]
CAPEX_power_capacity = (PCS + grid_connection) * summary["P_capacity_kW"]
EPC = (CAPEX_energy_capacity + CAPEX_power_capacity) * (10/100) # 10% of CAPEX -- 2025 EUR
CAPEX = CAPEX_energy_capacity + CAPEX_power_capacity + EPC

# O&M
Fixed_O_and_M = (1.43) * (summary["P_capacity_kW"]/summary["E_capacity_kWh"]) + (2.14) # % of CAPEX

pricing = summary[["E_capacity_kWh", "P_capacity_kW"]]
pricing["CAPEX_EUR"] = CAPEX
pricing["Fixed_O&M_EUR"] = CAPEX * (Fixed_O_and_M/100)
pricing["Total_Costs_EUR"] = CAPEX + Fixed_O_and_M
pricing.round(2)

### $\text{\textcolor{green}{Profile of Average Day}}$

In [ ]:
feasible = optimization_results[
    optimization_results["is_feasible"]
].copy()

def avg_profile(df, col):
    valid = df[col].dropna()
    valid = valid[valid.apply(lambda x: isinstance(x, (list, np.ndarray)))]
    return np.stack(valid.values).mean(axis=0)

time = np.arange(48) * 0.5

avg_load = avg_profile(feasible, "Pload_profile")
avg_gen = avg_profile(feasible, "Pgen_profile")
avg_pv = avg_profile(feasible, "PV_profile")
avg_wind = avg_profile(feasible, "Wind_profile")
avg_grid = avg_profile(feasible, "Pgrid_profile")
avg_curtail = avg_profile(feasible, "Pcurtail_profile")
avg_soc = avg_profile(feasible, "SOC_profile")
avg_charge = avg_profile(feasible, "Pcharge_profile")
avg_discharge = avg_profile(feasible, "Pdischarge_profile")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(time, avg_load, label="Load")
axes[0].plot(time, avg_gen, label="Total generation")
axes[0].plot(time, avg_pv, label="PV")
axes[0].plot(time, avg_wind, label="Wind")
axes[0].plot(time, avg_grid, label="Grid import")
axes[0].plot(time, avg_curtail, label="Curtailment")
axes[0].set_ylabel("Power [kW]")
axes[0].set_title("Average daily power flows")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(time, avg_soc, label="State of charge")
axes[1].plot(time, avg_charge, label="Battery charge")
axes[1].plot(time, avg_discharge, label="Battery discharge")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Energy [kWh] / Power [kW]")
axes[1].set_title("Average battery operation")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### $\text{\textcolor{green}{Plot for Best Battery Size}}$

$\text{\textcolor{green}{Optimal Battery Sizing Distribution}}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(feasible["E_capacity_kWh"].dropna(), bins=25)
axes[0].set_title("Optimal energy capacity")
axes[0].set_xlabel("Energy capacity [kWh]")
axes[0].set_ylabel("Number of days")
axes[0].grid(True)

axes[1].hist(feasible["P_capacity_kW"].dropna(), bins=25)
axes[1].set_title("Optimal power capacity")
axes[1].set_xlabel("Power capacity [kW]")
axes[1].set_ylabel("Number of days")
axes[1].grid(True)

plt.tight_layout()
plt.show()

$\text{\textcolor{green}{Operational Performance Related to Battery Energy Capacity}}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(
    feasible["E_capacity_kWh"],
    feasible["renewable_generation_pct"], # generation / load * 100
    alpha=0.7
)

axes[0].set_title("Battery capacity vs self-sufficiency")
axes[0].set_xlabel("Energy capacity [kWh]")
axes[0].set_ylabel("Share from Renewable Generation [%]")
axes[0].grid(True)
axes[0].set_ylim(0, 100)

axes[1].scatter(
    feasible["E_capacity_kWh"],
    feasible["grid_import_kWh"],
    alpha=0.7
)

axes[1].set_title("Battery capacity vs grid import")
axes[1].set_xlabel("Energy capacity [kWh]")
axes[1].set_ylabel("Grid import [kWh/day]")
axes[1].grid(True)

plt.tight_layout()
plt.show()


$\text{\textcolor{green}{Optimization Across All Paired Days}}$

$\text{Check whether this makes more sense given that charge of battery should be allowed to be carried from one day to another.}$
$\text{You wouldn't be optimizating on a day-by-day basis anymore, since it's very hard for the self-sufficiency requirement to be met every day in the 323 days}$

In [ ]:
def solve_battery_linked_days(
    paired_day_profiles,
    minimum_self_sufficiency,
    charge_eff,
    discharge_eff,
    dT,
    C_rate
):
    # Flatten all daily profiles into one continuous time series
    Pload_np = np.concatenate(paired_day_profiles["Load_profile"].to_numpy())
    PV_gen = np.concatenate(paired_day_profiles["PV_profile"].to_numpy())
    Wind_gen = np.concatenate(paired_day_profiles["Wind_profile"].to_numpy())
    Pprice_np = np.concatenate(paired_day_profiles["Price_profile"].to_numpy())

    Pgen_total_np = PV_gen + Wind_gen
    n = len(Pload_np)

    # Input checks
    if not np.isfinite(Pload_np).all():
        return {"status": "bad_input: non-finite load"}
    if not np.isfinite(PV_gen).all():
        return {"status": "bad_input: non-finite PV"}
    if not np.isfinite(Wind_gen).all():
        return {"status": "bad_input: non-finite wind"}
    if not np.isfinite(Pprice_np).all():
        return {"status": "bad_input: non-finite price"}

    M = 1.1 * max(np.max(Pload_np), np.max(Pgen_total_np))

    # Constants
    Pload = cp.Constant(Pload_np)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)

    # Variables
    P_capacity = cp.Variable(nonneg=True)
    E_capacity = cp.Variable(nonneg=True)

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)
    Pgrid_buy = cp.Variable(n, nonneg=True)
    Pcurtail = cp.Variable(n, nonneg=True)
    SOC = cp.Variable(n)

    total_load_kWh = np.sum(Pload_np) * dT

    # Removed this:
    #and M to cap P_capacity
    # z = cp.Variable(n, boolean=True)

    constraints = [
        P_capacity <= C_rate * E_capacity,

        Pcharge <= P_capacity,
        Pdischarge <= P_capacity,

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        Pcurtail <= Pgen_PV + Pgen_W,

        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy
        == Pload + Pcharge + Pcurtail,

        SOC[0] == 0.5 * E_capacity
        + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,

        SOC[1:] == SOC[:-1]
        + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[-1] >= 0.5 * E_capacity,

        cp.sum(Pgrid_buy) * dT
        <= (1 - minimum_self_sufficiency) * total_load_kWh
    ]

    E_grid_cost = cp.sum(cp.multiply(Pgrid_buy, Pprice_np)) * dT

    lambda_curt = 1000
    lambda_throughput = 3000

    E_curtailed = lambda_curt * cp.sum(Pcurtail) * dT

    E_charge_discharge = (lambda_throughput * cp.sum(Pcharge + Pdischarge) * dT)

    problem = cp.Problem(
        cp.Minimize(E_capacity + E_grid_cost + E_curtailed + E_charge_discharge), constraints)
    
    try:
        problem.solve(solver=cp.HIGHS, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            return {
                "status": problem.status,
                "E_capacity_kWh": np.nan,
                "P_capacity_kW": np.nan,
            }

        total_generation_kWh = np.sum(Pgen_total_np) * dT
        grid_import_kWh = np.sum(Pgrid_buy.value) * dT
        spilled_energy_kWh = np.sum(Pcurtail.value) * dT
        simultaneous_charge_discharge = np.any(
        (Pcharge.value > 1e-6) & (Pdischarge.value > 1e-6))
        simultaneous_charge_discharge_kWh = (np.sum(np.minimum(Pcharge.value, Pdischarge.value)) * dT)

        return {
            "status": problem.status,
            "E_capacity_kWh": E_capacity.value,
            "P_capacity_kW": P_capacity.value,
            
            "simultaneous_charge_discharge": simultaneous_charge_discharge,
            "simultaneous_charge_discharge_kWh": simultaneous_charge_discharge_kWh,
            "storage_duration_hrs": E_capacity.value / P_capacity.value,

            "total_generation_kWh": total_generation_kWh,
            "total_load_kWh": total_load_kWh,
            "grid_import_kWh": grid_import_kWh,
            "grid_dependence_pct": grid_import_kWh / total_load_kWh * 100,
            "achieved_self_sufficiency_pct": (1 - grid_import_kWh / total_load_kWh) * 100,
            "spilled_energy_kWh": spilled_energy_kWh,

            "Pload_profile": Pload_np,
            "Pgen_profile": Pgen_total_np,
            "PV_profile": PV_gen,
            "Wind_profile": Wind_gen,
            "Pcharge_profile": Pcharge.value,
            "Pdischarge_profile": Pdischarge.value,
            "Pgrid_profile": Pgrid_buy.value,
            "Pcurtail_profile": Pcurtail.value,
            "SOC_profile": SOC.value,
        }

    except Exception as e:
        return {
            "status": f"solver_error: {type(e).__name__}: {str(e)}",
            "E_capacity_kWh": np.nan,
            "P_capacity_kW": np.nan,
        }

In [ ]:
targets = np.linspace(0.1, 1.0, 10)

results = []

for target in targets:
    out = solve_battery_linked_days(
        paired_day_profiles=paired_day_profiles,
        minimum_self_sufficiency=target,
        charge_eff=0.95,
        discharge_eff=0.95,
        dT=0.5,
        C_rate=0.5
    )

    out["target_self_sufficiency"] = target
    out["C_rate"] = 0.5
    out["is_feasible"] = out["status"] in ["optimal", "optimal_inaccurate"]

    results.append(out)

linked_optimization_results = pd.DataFrame(results)

In [ ]:
linked_optimization_results["storage_duration_hrs"] = (
    linked_optimization_results["E_capacity_kWh"]
    / linked_optimization_results["P_capacity_kW"]
)

summary = linked_optimization_results[
    [
        "target_self_sufficiency",
        "status",
        "E_capacity_kWh",
        "P_capacity_kW",
        "storage_duration_hrs",
        "total_generation_kWh",
        "total_load_kWh",
        "grid_import_kWh",
        "grid_dependence_pct",
        "achieved_self_sufficiency_pct",
        "spilled_energy_kWh",
        "simultaneous_charge_discharge",
        "simultaneous_charge_discharge_kWh"
    ]
]

summary["target_self_sufficiency"] = (
    summary["target_self_sufficiency"] * 100
).round(0).astype(int)

summary = summary.round(2)

summary